# 25｜附錄B_混合精度與量化

執行對稱 int8 量化，直接比較原權重、整數碼與反量化誤差。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])


In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")


## 課堂小實驗

In [ ]:
weights = np.array([[-1.20, -0.35, 0.0, 0.42, 1.05], [0.18, -0.77, 0.63, 0.91, -0.12]])
scale = np.max(np.abs(weights)) / 127
q = np.clip(np.round(weights / scale), -127, 127).astype(np.int8)
dequant = q.astype(float) * scale
err = dequant - weights
table = pd.DataFrame({'weight': weights.ravel(), 'int8': q.ravel(),
                      'dequantized': dequant.ravel(), 'error': err.ravel()})
print('scale:', scale, ' max abs error:', np.max(np.abs(err)))
table.round(5)

## 執行結果圖

In [ ]:
x = np.arange(weights.size)
plt.plot(x, weights.ravel(), 'o-', label='float')
plt.plot(x, dequant.ravel(), 'x--', label='dequantized int8')
plt.xlabel('weight index'); plt.ylabel('value'); plt.title('Symmetric int8 quantization')
plt.legend()
savefig('25_quantization_demo')
report('quantization', {'scale': scale, 'max_abs_error': np.max(np.abs(err)), 'codes': q.tolist()})

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。